In [1]:
library(dplyr)
library(tidyr)
library(pheatmap)
library(viridis)
library(tools)
library(cluster)
library(igraph)
library(tibble)


Warning message:
“程辑包‘dplyr’是用R版本4.2.3 来建造的”

载入程辑包：‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“程辑包‘tidyr’是用R版本4.2.3 来建造的”
Warning message:
“程辑包‘viridis’是用R版本4.2.3 来建造的”
载入需要的程辑包：viridisLite

Warning message:
“程辑包‘viridisLite’是用R版本4.2.3 来建造的”
Warning message:
“程辑包‘cluster’是用R版本4.2.3 来建造的”

载入程辑包：‘igraph’


The following object is masked from ‘package:tidyr’:

    crossing


The following objects are masked from ‘package:dplyr’:

    as_data_frame, groups, union


The following objects are masked from ‘package:stats’:

    decompose, spectrum


The following object is masked from ‘package:base’:

    union



载入程辑包：‘tibble’


The following object is masked from ‘package:igraph’:

    as_data_frame




In [2]:
# 读入GEP30个，没有过滤
signature_Neutro <- read.csv(
  "Example_refbuilder_Neutrophils/ALL_cGEP_top_genes_Neutrophils_wide.csv",
  check.names = FALSE,
  stringsAsFactors = FALSE)
# 转成 list（核心）
cluster_genes <- lapply(signature_Neutro, function(x) {
  x[!is.na(x) & x != ""]
})


gene_df <- data.frame(
  cGEP_Cluster = names(cluster_genes),  # 确保这里是 cGEP1, cGEP2...
  cGEP_signature = sapply(cluster_genes, function(x) paste(x, collapse = ",")),
  stringsAsFactors = FALSE
)
gene_df

,cGEP_Cluster,cGEP_signature
,<chr>,<chr>
cGEP1,cGEP1,"CD3G,IL32,LCK,CCL5,CD3E,GZMA,NKG7,CD2,TRBC2,ITM2A,CD3D,CD27,TRAC,IKZF3,GZMH,RASGRP1,TRAT1,CTLA4,ETS1,GZMK,CD8B,CD8A,SYNE2,TRBC1,CTSW,CST7,LINC00426,CD7,PRF1,SKAP1"
cGEP2,cGEP2,"CXCR5,LINC01641,GAS5,RPS8,RPS18,RPS2,RPL37A,RPLP2,RPL23A,RPS23,RPS12,RPL18A,RPL10A,RPL3,RPS27,RPS29,RPL13,RPL13A,EEF1A1,RPS6,RPL41,RPL10,TRBC1,RPS3,RPL4,RPS17,SNHG6,RPS20,RPS14,RPL19"
cGEP3,cGEP3,"S100A9,S100A12,S100A8,VCAN,LYZ,PLBD1,VIM,SERPINB1,METTL9,HP,TKT,MNDA,PADI4,GCA,GAPDH,APLP2,SLC2A3,ITGAM,IRS2,IVNS1ABP,CKAP4,HMGB2,MXD1,CES1,TSPO,GM2A,TALDO1,ANXA1,SELL,RBP7"
cGEP4,cGEP4,"IER3,IL1B,NFKBIA,NLRP3,AC095055.1,TNFAIP3,VEGFA,KDM6B,ZNF331,INSIG1,GPR183,MAP3K8,AC092368.3,EREG,CXCL2,GRASP,CXCL8,AC016831.7,RASGEF1B,PLAUR,NR4A1,THBS1,NAMPT,CD83,ARF4-AS1,CLHC1,HLA-DQB2,PIM3,MRPS24,CSRNP1"
cGEP5,cGEP5,"IFIT1,RSAD2,IFIT3,ISG15,MX1,HERC5,IFIT2,GBP1,CMPK2,SPDEF,AC010501.1,OAS2,AC138907.8,IFI44L,SERPING1,IFITM3,GBP5,GBP4,IFI6,AC007362.1,EPSTI1,RNF213,DDX58,AC148477.3,UBE2L6,XAF1,STAT1,AC113189.2,LINC02453,TNFSF13B"
cGEP6,cGEP6,"FCGR3A,RYR3,AIF1,C1QA,LST1,IFITM3,RHOC,SMIM25,CDKN1C,C1QB,FCER1G,CLIC2,CD79B,ABI3,CSF1R,AC007038.1,TCF7L2,ATP8B1,FMNL2,CEBPB,TRNP1,LINC02449,DRAP1,AC124045.1,COTL1,MTSS1,LRRC25,L1TD1,RPS19,C1QC"
cGEP7,cGEP7,"HLA-DQA1,FCER1A,HLA-DRA,HLA-DPA1,HLA-DQA2,RGS1,HLA-DPB1,HLA-DQB1,HLA-DRB1,HLA-DRB5,CLEC10A,CD74,HIST1H3A,HLA-DMA,CD1C,CCSER1,AC102953.2,AL390728.5,ZNF697,MRC1,ZNF790,HLA-DOA,TNFRSF21,LINC01970,AC124068.2,CST7,SH3BP4,ZNF597,TNFRSF12A,RGL1"
cGEP8,cGEP8,"BCL2,AL049840.1,LOH12CR2,AC092368.3,RILPL1,CCNH,CDKN2A,SNX24,AL117339.5,YES1,ACVRL1,AL450326.1,POLR3B,CDKL3,AL138820.1,TUBGCP5,AC119428.2,AL590617.2,ZNF517,NPM3,HOXB3,CAMK4,CNTNAP2,HYPK,AC124242.1,TFRC,IKZF4,AC114490.2,TMEM236,AC005838.2"
cGEP9,cGEP9,"PLK2,MSR1,C3,ACVRL1,ACKR3,APOC1,ZNF707,SPARC,PHLDA1,AC011043.1,CD81,LTC4S,TBX6,ZBED3-AS1,AC005076.1,LINC01431,NRP1,OSM,AL357060.1,SLC2A5,PSD,MARCO,WDR76,AL450326.1,GPT2,OLR1,UBE3D,FN1,LINC01678,EPS8L1"


In [4]:
#Neu_gep_anno <- read.csv('./3.Neutrophils_GEP_Anno/3.3.Neutrophils_GEP30_Anno_final.csv')
Neu_gep_anno <- read.csv('/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.6.cNMF_menchmark/cGEP_signature/Neutrophils_cGEP30.csv')
Neu_gep_anno

cGEP_Cluster,cGEP_Anno_Name,Category
<chr>,<chr>,<chr>
cGEP5,Neu-IFN,Functional
cGEP18,Neu-Degranulation,Functional
cGEP16,Art-Mito,Artifact
cGEP4,Neu-Inflammation,Functional
cGEP6,Doublet-Mono,Doublet
cGEP27,Art-Stress,Artifact
cGEP3,Doublet-Mono,Doublet
cGEP2,Art-Ribo,Artifact
cGEP25,Doublet-Mono,Doublet


In [5]:
final_combined_df <- Neu_gep_anno %>%
  left_join(gene_df, by = "cGEP_Cluster")
final_combined_df

cGEP_Cluster,cGEP_Anno_Name,Category,cGEP_signature
<chr>,<chr>,<chr>,<chr>
cGEP5,Neu-IFN,Functional,"IFIT1,RSAD2,IFIT3,ISG15,MX1,HERC5,IFIT2,GBP1,CMPK2,SPDEF,AC010501.1,OAS2,AC138907.8,IFI44L,SERPING1,IFITM3,GBP5,GBP4,IFI6,AC007362.1,EPSTI1,RNF213,DDX58,AC148477.3,UBE2L6,XAF1,STAT1,AC113189.2,LINC02453,TNFSF13B"
cGEP18,Neu-Degranulation,Functional,"LCN2,RETN,LTF,CD24,CEACAM8,BPI,CAMP,CEACAM6,CRISP3,LYZ,OLFM4,RNASE2,TCN1,RNASE3,STOM,MS4A3,CEBPE,HTRA3,COL17A1,DEFA4,TFF3,LINC02009,ABCA13,CTSG,MMP8,ERG,AZU1,PRTN3,BEX1,C20orf27"
cGEP16,Art-Mito,Artifact,"MT-ATP6,MT-CO1,MT-ND4,MT-CO3,MT-CYB,MT-ND1,MT-ND3,MT-ATP8,MT-CO2,MT-ND5,MT-ND2,MT-ND4L,MIAT,VCAN,SPINK4,MUC5AC,MT-RNR2,PSCA,KRTAP4-8,CDC42BPG,PHF10,PLXNB2,ADO,MARCH1,SH3BP4,MT-RNR1,MALAT1,RNF217,TMEM190,LINC00315"
cGEP4,Neu-Inflammation,Functional,"IER3,IL1B,NFKBIA,NLRP3,AC095055.1,TNFAIP3,VEGFA,KDM6B,ZNF331,INSIG1,GPR183,MAP3K8,AC092368.3,EREG,CXCL2,GRASP,CXCL8,AC016831.7,RASGEF1B,PLAUR,NR4A1,THBS1,NAMPT,CD83,ARF4-AS1,CLHC1,HLA-DQB2,PIM3,MRPS24,CSRNP1"
cGEP6,Doublet-Mono,Doublet,"FCGR3A,RYR3,AIF1,C1QA,LST1,IFITM3,RHOC,SMIM25,CDKN1C,C1QB,FCER1G,CLIC2,CD79B,ABI3,CSF1R,AC007038.1,TCF7L2,ATP8B1,FMNL2,CEBPB,TRNP1,LINC02449,DRAP1,AC124045.1,COTL1,MTSS1,LRRC25,L1TD1,RPS19,C1QC"
cGEP27,Art-Stress,Artifact,"HSPA1A,JUN,KLF2,DNAJA4,IER5,HSPD1,HSPA8,BAG3,HSP90AA1,DNAJB1,HSP90AB1,RHOB,HSPH1,DNAJA1,KLF4,PPP1R15A,CLK1,HSPE1,UBC,TRA2B,ZFAND2A,EIF4A3,HSPA6,CHORDC1,KLF6,GADD45B,JUND,DNAJB6,BTG2,HSPA1B"
cGEP3,Doublet-Mono,Doublet,"S100A9,S100A12,S100A8,VCAN,LYZ,PLBD1,VIM,SERPINB1,METTL9,HP,TKT,MNDA,PADI4,GCA,GAPDH,APLP2,SLC2A3,ITGAM,IRS2,IVNS1ABP,CKAP4,HMGB2,MXD1,CES1,TSPO,GM2A,TALDO1,ANXA1,SELL,RBP7"
cGEP2,Art-Ribo,Artifact,"CXCR5,LINC01641,GAS5,RPS8,RPS18,RPS2,RPL37A,RPLP2,RPL23A,RPS23,RPS12,RPL18A,RPL10A,RPL3,RPS27,RPS29,RPL13,RPL13A,EEF1A1,RPS6,RPL41,RPL10,TRBC1,RPS3,RPL4,RPS17,SNHG6,RPS20,RPS14,RPL19"
cGEP25,Doublet-Mono,Doublet,"C1QB,C1QC,C1QA,TMEM176B,MS4A6A,MS4A4A,ITM2B,NPC2,FCGRT,IL2RA,FUCA1,PLTP,FOLR2,TSPAN4,CD74,HLA-C,CST3,CTSC,HLA-DMB,VSIG4,TMEM176A,GPR34,HLA-DMA,APOE,MGST2,SLC40A1,GRN,ADAMDEC1,CTSZ,DNASE2"


In [8]:
#OUT_FILE <- "./3.Neutrophils_GEP_Anno/3.3.Neutrophil_cGEP30_Anno_Complete_With_Genes.csv"
OUT_FILE <- '/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.6.cNMF_menchmark/cGEP_signature/Neutrophils_cGEP30_signature.csv'

write.csv(final_combined_df, OUT_FILE, row.names = FALSE)